In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit
import boto3

In [6]:
MINIO_ACCESS_KEY= "minioadmin"
MINIO_SECRET_KEY= "minioadmin"
MINIO_ENDPOINT = "http://minio:9000"
BUCKET_NAME= "datalake"
LOCAL_DATA_PATH= "/raw_mount"

In [7]:
spark= SparkSession.builder.appName("processVariant") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .getOrCreate()

In [8]:
interactions= spark.read.parquet("s3a://datalake/raw_parquet/interactions_parquet")

25/11/27 05:08:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [9]:
interactions.columns

['gene_claim_name',
 'gene_concept_id',
 'gene_name',
 'interaction_source_db_name',
 'interaction_source_db_version',
 'interaction_type',
 'interaction_score',
 'drug_claim_name',
 'drug_concept_id',
 'drug_name',
 'approved',
 'immunotherapy',
 'anti_neoplastic']

In [13]:
interactions.select("gene_claim_name", "gene_concept_id", "gene_name").show(20, False)

+---------------+---------------+---------+
|gene_claim_name|gene_concept_id|gene_name|
+---------------+---------------+---------+
|DACC           |NULL           |NULL     |
|DACB           |NULL           |NULL     |
|FTSI           |NULL           |NULL     |
|MRCA           |NULL           |NULL     |
|MRCB           |NULL           |NULL     |
|DACA           |NULL           |NULL     |
|MRDA           |NULL           |NULL     |
|DACC           |NULL           |NULL     |
|DACB           |NULL           |NULL     |
|FTSI           |NULL           |NULL     |
|MRCA           |NULL           |NULL     |
|MRCB           |NULL           |NULL     |
|DACA           |NULL           |NULL     |
|MRDA           |NULL           |NULL     |
|DACC           |NULL           |NULL     |
|PBP4           |NULL           |NULL     |
|UNIPROT:P48048 |hgnc:6255      |KCNJ1    |
|PBPF           |NULL           |NULL     |
|UNIPROT:Q9NYX4 |hgnc:17938     |CALY     |
|MECA           |NULL           

In [14]:
interactions.select("interaction_source_db_name", "interaction_source_db_version", "interaction_type", "interaction_score").show(20, False)

+--------------------------+-----------------------------+----------------+-----------------+
|interaction_source_db_name|interaction_source_db_version|interaction_type|interaction_score|
+--------------------------+-----------------------------+----------------+-----------------+
|ChEMBL                    |33                           |inhibitor       |NULL             |
|ChEMBL                    |33                           |inhibitor       |NULL             |
|ChEMBL                    |33                           |inhibitor       |NULL             |
|ChEMBL                    |33                           |inhibitor       |NULL             |
|ChEMBL                    |33                           |inhibitor       |NULL             |
|ChEMBL                    |33                           |inhibitor       |NULL             |
|ChEMBL                    |33                           |inhibitor       |NULL             |
|ChEMBL                    |33                           |in

In [16]:
interactions.select("drug_claim_name", "drug_concept_id", "drug_name", "approved", "immunotherapy", "anti_neoplastic").show(20, False)

+--------------------+---------------+---------------------+--------+-------------+---------------+
|drug_claim_name     |drug_concept_id|drug_name            |approved|immunotherapy|anti_neoplastic|
+--------------------+---------------+---------------------+--------+-------------+---------------+
|CEFTAZIDIME         |rxcui:1545984  |CEFTAZIDIME ANHYDROUS|TRUE    |FALSE        |FALSE          |
|CEFPODOXIME PROXETIL|rxcui:47835    |CEFPODOXIME PROXETIL |TRUE    |FALSE        |FALSE          |
|CEFPODOXIME PROXETIL|rxcui:47835    |CEFPODOXIME PROXETIL |TRUE    |FALSE        |FALSE          |
|CEFPODOXIME PROXETIL|rxcui:47835    |CEFPODOXIME PROXETIL |TRUE    |FALSE        |FALSE          |
|CEFPODOXIME PROXETIL|rxcui:47835    |CEFPODOXIME PROXETIL |TRUE    |FALSE        |FALSE          |
|CEFPODOXIME PROXETIL|rxcui:47835    |CEFPODOXIME PROXETIL |TRUE    |FALSE        |FALSE          |
|CEFPODOXIME PROXETIL|rxcui:47835    |CEFPODOXIME PROXETIL |TRUE    |FALSE        |FALSE          |


In [17]:
human_intrct= interactions.filter(interactions.gene_concept_id.startswith("hgnc:"))

In [20]:
human_intrct.limit(20).toPandas()

,gene_claim_name,gene_concept_id,gene_name,interaction_source_db_name,interaction_source_db_version,interaction_type,interaction_score,drug_claim_name,drug_concept_id,drug_name,approved,immunotherapy,anti_neoplastic
0,UNIPROT:P48048,hgnc:6255,KCNJ1,TdgClinicalTrial,Jan-14,NULL,1.093909817410302,TOLBUTAMIDE,rxcui:10635,TOLBUTAMIDE,TRUE,FALSE,FALSE
1,UNIPROT:Q9NYX4,hgnc:17938,CALY,TdgClinicalTrial,Jan-14,NULL,1.750255707856483,TRIFLUOPERAZINE,rxcui:10800,TRIFLUOPERAZINE,TRUE,FALSE,FALSE
2,UNIPROT:P18825,hgnc:283,ADRA2C,TdgClinicalTrial,Jan-14,NULL,0.1305078821102597,YOHIMBINE,rxcui:220982,YOHIMBINE,TRUE,FALSE,FALSE
3,UNIPROT:P37288,hgnc:895,AVPR1A,TdgClinicalTrial,Jan-14,NULL,0.3512218811752141,VASOPRESSIN,rxcui:1098,ARGIPRESSIN,TRUE,FALSE,FALSE
4,ENV,hgnc:39031,ERVK-20,ChEMBL,33,inhibitor,5.250767123569449,ENFUVIRTIDE,rxcui:139896,ENFUVIRTIDE,TRUE,FALSE,FALSE
5,UNIPROT:P30518,hgnc:897,AVPR2,TdgClinicalTrial,Jan-14,NULL,0.3671865121377238,VASOPRESSIN,rxcui:1098,ARGIPRESSIN,TRUE,FALSE,FALSE
6,UNIPROT:P30518,hgnc:897,AVPR2,TEND,1-Aug-11,NULL,1.790034246671403,CONIVAPTAN,rxcui:302285,CONIVAPTAN,TRUE,FALSE,FALSE
7,RPMJ,hgnc:14490,MRPL36,ChEMBL,33,inhibitor,0.3860858179095183,STREPTOMYCIN SULFATE,rxcui:10110,STREPTOMYCIN SULFATE,TRUE,FALSE,FALSE
8,UNIPROT:P28476,hgnc:4091,GABRR2,TEND,1-Aug-11,NULL,0.2375912725597036,ADINAZOLAM,ncit:C76531,ADINAZOLAM,FALSE,FALSE,FALSE
9,UNIPROT:P14867,hgnc:4075,GABRA1,TEND,1-Aug-11,NULL,0.055504938,CLOBAZAM,rxcui:21241,CLOBAZAM,TRUE,FALSE,FALSE


In [24]:
print(human_intrct.count())
print(human_intrct.filter((human_intrct.gene_concept_id.isNotNull())).count())
print(human_intrct.filter((human_intrct.gene_claim_name.isNotNull())).count())

88976
88976
88976


In [26]:
print(human_intrct.select("gene_concept_id").distinct().count())
print(human_intrct.select("gene_claim_name").distinct().count())

4877
7822


In [28]:
human_intrct.distinct().count()

88976

In [29]:
human_intrct.select("gene_concept_id", "drug_concept_id").distinct().count()

69831

25/11/27 06:21:25 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-b9d4aefb-9a78-43b3-8d2d-b2e8e060580f. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-b9d4aefb-9a78-43b3-8d2d-b2e8e060580f
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:173)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:364)
	at scala.collection.IndexedSeqOptimize

In [4]:
spark.stop()